# Ropedia Academy — B8 · Dynamic & 4D reconstruction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B8.ipynb)

> **Factors a dynamic scene into a shared canonical model + a time deformation field, then adds the as-rigid-as-possible prior that tames the under-constrained 4D problem.**
>
> 把动态场景分解为共享标准模型 + 时间形变场，再加入尽可能刚性先验，驯服欠约束的 4D 问题。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B8

In [ ]:
import torch, torch.nn as nn

# 4D = a CANONICAL model (rest scene) + a DEFORMATION field warping time t into it.
canonical = nn.Sequential(nn.Linear(3,64), nn.ReLU(), nn.Linear(64,4))   # shared over time
deform    = nn.Sequential(nn.Linear(4,64), nn.ReLU(), nn.Linear(64,3))   # (x,t) -> delta x

def render_at(x, t):
    dx = deform(torch.cat([x, torch.full_like(x[:, :1], t)], -1))   # warp to canonical
    return canonical(x + dx), dx

x = torch.randn(100, 3, requires_grad=True)
out0, d0 = render_at(x, 0.0)
out1, d1 = render_at(x, 1.0)                          # SAME appearance model, new motion
print("appearance shared, only motion differs:", d0.abs().mean().item() != d1.abs().mean().item())

# Under-constrained -> need priors. As-rigid-as-possible: neighbours deform alike.
nbr  = torch.cdist(x, x).topk(4, largest=False).indices    # 3 nearest neighbours
arap = (d1[nbr] - d1[:, None]).pow(2).mean()               # local-rigidity penalty
print("as-rigid-as-possible regularizer:", round(arap.item(), 4))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B8
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks